# SeqMat vs Biopython — a side-by-side

Biopython is the long-standing default for biological-sequence work in Python. SeqMat is newer and narrower. The two are designed around different shapes of data:

- **Biopython** stores a sequence as a string-like object. Operations on the bases are C-fast.
- **SeqMat** stores a sequence as a NumPy structured array — `nt`, `ref`, `index`, `mut_type`, `valid` in parallel columns — so every base carries its genomic coordinate and original reference.

This notebook runs the same task in both libraries, side by side, on real data. Each section answers one question: *for this workload, which is the right tool?*

All timings are on an M-series Mac, one core, warm caches. Numbers are minimum-of-5 wall-clock seconds to suppress GC noise.

In [1]:
import random, time
from Bio.Seq import Seq, MutableSeq
from seqmat import SeqMat, Gene

def best_of(fn, runs=5):
    best = float('inf')
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best

## 1. Reverse-complement a 1 Mb sequence

The simplest comparison. No coordinates, no history — just flip the bases.

In [2]:
rng = random.Random(0)
seq_1m = ''.join(rng.choices('ACGT', k=1_000_000))

def bio_rc():    return str(Seq(seq_1m).reverse_complement())
def seqmat_rc(): s = SeqMat(seq_1m); s.reverse_complement(); return s.seq

t_bio    = best_of(bio_rc)
t_seqmat = best_of(seqmat_rc)

print(f'Biopython Seq.reverse_complement():  {t_bio*1e3:6.2f} ms')
print(f'SeqMat.reverse_complement():         {t_seqmat*1e3:6.2f} ms  ({t_seqmat/t_bio:.1f}x)')
print(f'Outputs match: {bio_rc() == seqmat_rc()}')

Biopython Seq.reverse_complement():    0.57 ms
SeqMat.reverse_complement():          17.50 ms  (30.9x)
Outputs match: True


Biopython wins. The reason is structural: SeqMat is carrying a structured array (5 columns, 27 bytes per record) and physically reversing it on every call. Biopython is just translating + reversing bytes. If reverse-complement on long sequences is your hot loop and you don't need the coordinate bookkeeping, use Biopython.

## 2. Apply 1,000 SNPs to a 45 kb sequence

Same question, harder workload: mutate a thousand individual positions. SeqMat records every change to `s.mutations`; Biopython's `MutableSeq[p] = b` doesn't track anything by default.

In [3]:
rng = random.Random(0)
seq_str = 'ATCG' * 11_250
N = 1000
zero_pos  = sorted(random.sample(range(len(seq_str)), N))
new_bases = [rng.choice('ACGT') for _ in range(N)]
one_pos   = [p + 1 for p in zero_pos]               # SeqMat is 1-indexed by default
muts      = list(zip(one_pos, [seq_str[p] for p in zero_pos], new_bases))

def bio_apply():
    s = MutableSeq(seq_str)
    for p, b in zip(zero_pos, new_bases):
        s[p] = b
    return str(s)

def seqmat_apply():
    s = SeqMat(seq_str)
    s.apply_mutations(muts, permissive_ref=True)
    return s.seq

t_bio    = best_of(bio_apply)
t_seqmat = best_of(seqmat_apply) - best_of(lambda: SeqMat(seq_str))
print(f'Biopython MutableSeq (no history):       {t_bio*1e3:6.2f} ms')
print(f'SeqMat apply_mutations (with history):   {t_seqmat*1e3:6.2f} ms  ({t_seqmat/t_bio:.1f}x)')
print(f'Outputs match: {bio_apply() == seqmat_apply()}')

Biopython MutableSeq (no history):         0.21 ms
SeqMat apply_mutations (with history):     0.56 ms  (2.7x)
Outputs match: True


Biopython is still faster, but the gap is now small — SeqMat is doing extra work (recording every mutation, validating refs, maintaining `mut_type` and `mutated_positions`) that Biopython is not. If you want any of that, you'd add it on top of `MutableSeq` yourself.

## 3. "What changed?" — mutation history

Now ask: *after applying the same 1,000 SNPs, how do I know which positions changed and from what to what?*

In SeqMat the answer is one attribute access. In Biopython you'd have to record it yourself as you applied each change.

In [4]:
# SeqMat: history is recorded automatically.
s = SeqMat(seq_str)
s.apply_mutations(muts, permissive_ref=True)
print('SeqMat — first 3 recorded mutations:')
for m in s.mutations[:3]:
    print(' ', m)
print(f'Total recorded: {len(s.mutations)}')

SeqMat — first 3 recorded mutations:
  {'pos': 17, 'ref': 'A', 'alt': 'T', 'type': 'snp'}
  {'pos': 41, 'ref': 'A', 'alt': 'T', 'type': 'snp'}
  {'pos': 48, 'ref': 'G', 'alt': 'A', 'type': 'snp'}
Total recorded: 748


In [5]:
# Biopython equivalent: layer your own history on top.
s = MutableSeq(seq_str)
history = []
for p, b in zip(zero_pos, new_bases):
    ref = s[p]
    if ref != b:
        history.append({'pos': p, 'ref': ref, 'alt': b, 'type': 'snp'})
        s[p] = b
print('Biopython + hand-rolled history — first 3:')
for m in history[:3]:
    print(' ', m)
print(f'Total recorded: {len(history)}')

Biopython + hand-rolled history — first 3:
  {'pos': 16, 'ref': 'A', 'alt': 'T', 'type': 'snp'}
  {'pos': 40, 'ref': 'A', 'alt': 'T', 'type': 'snp'}
  {'pos': 47, 'ref': 'G', 'alt': 'A', 'type': 'snp'}
Total recorded: 748


Both paths arrive at the same answer. The difference is who writes the bookkeeping — SeqMat does it for you; with Biopython it's on the caller. Either is fine; pick by how often you actually need the history.

## 4. Splice introns, keep coordinates

This is where the two libraries genuinely diverge. 

Suppose you have a pre-mRNA with three intron spans `[(10, 20), (30, 40), (50, 60)]` and you want the mature mRNA. After splicing, what's at *original* coordinate 45?

- **SeqMat:** `s[45]` still works after `s.remove_regions(...)` because the surviving rows keep their original `index`.
- **Biopython:** `Seq` slicing renumbers from 0. You have to maintain your own offset table to map old→new positions.

In [6]:
import numpy as np

# A 70-base pre-mRNA, three introns spliced out.
seq_str = ('A' * 10) + ('C' * 11) + ('G' * 9) + ('T' * 11) + ('A' * 9) + ('C' * 11) + ('G' * 9)
introns = [(10, 20), (30, 40), (50, 60)]

# SeqMat — coordinates are preserved.
s = SeqMat(seq_str)            # 1-indexed: positions 1..70
spliced = s.remove_regions(introns)
print('SeqMat: base at original position 45 after splicing:', spliced[45]['nt'].decode())
print('SeqMat: spliced sequence:', spliced.seq)

SeqMat: base at original position 45 after splicing: A
SeqMat: spliced sequence: AAAAAAAAACGGGGGGGGTAAAAAAAACGGGGGGGGG


In [7]:
# Biopython — you splice the string and maintain your own coordinate map.
def biopython_splice_with_map(seq, introns):
    keep_parts = []
    keep_indices = []   # original 1-based position for each surviving base
    last = 0
    for a, b in introns:
        keep_parts.append(str(seq[last:a-1]))               # half-open / 0-based slicing
        keep_indices.extend(range(last + 1, a))             # 1-based originals
        last = b                                            # next exon starts at b+1 → 0-based b
    keep_parts.append(str(seq[last:]))
    keep_indices.extend(range(last + 1, len(seq) + 1))
    return ''.join(keep_parts), keep_indices

spliced_str, orig_positions = biopython_splice_with_map(Seq(seq_str), introns)
# Look up base at original position 45.
bio_idx = orig_positions.index(45)
print('Biopython: base at original position 45 after splicing:', spliced_str[bio_idx])
print('Biopython: spliced sequence:', spliced_str)

Biopython: base at original position 45 after splicing: A
Biopython: spliced sequence: AAAAAAAAACGGGGGGGGTAAAAAAAACGGGGGGGGG


Same answer either way. With SeqMat the call is one line and the coordinate semantics are built into the data model. With Biopython the surrounding bookkeeping is the bulk of the code, and it's easy to get the half-open vs. inclusive boundaries wrong (the cell above has a few easy off-by-ones to trip over).

If your pipeline keeps coming back to *"what was at chrN:position before this transformation?"*, SeqMat saves you from writing that table by hand. If you only ever look at the spliced sequence, Biopython is fine.

## 5. End-to-end: mutate a real gene at a genomic coordinate, see the protein effect

The KRAS G12D mutation is `chr12:25,245,350 C>T` on the reference (forward) strand. KRAS is on the minus strand, so the mRNA reads `GGT → GAT` (Gly → Asp) at codon 12.

Both libraries can produce the same answer. The amount of code is different.

In [8]:
# SeqMat path
kras = Gene.from_position('12', 25_245_350)[0]   # locate the gene from the coordinate
tx_mut = kras.transcript()
tx_mut.generate_pre_mrna()
tx_mut.pre_mrna.apply_mutations([(25_245_350, 'C', 'T')])
tx_mut.generate_mature_mrna()
tx_mut.generate_protein()

tx_wt = kras.transcript()
tx_wt.generate_mature_mrna()
tx_wt.generate_protein()

wt, mut = tx_wt.protein, tx_mut.protein
diffs = [(i+1, a, b) for i, (a, b) in enumerate(zip(wt, mut)) if a != b]
print('SeqMat: WT  protein[:25]:', wt[:25])
print('SeqMat: MUT protein[:25]:', mut[:25])
print('SeqMat: differences:     ', diffs)

SeqMat: WT  protein[:25]: MTEYKLVVVGAGGVGKSALTIQLIQ
SeqMat: MUT protein[:25]: MTEYKLVVVGADGVGKSALTIQLIQ
SeqMat: differences:      [(12, 'G', 'D')]


The equivalent in Biopython needs you to do everything SeqMat did implicitly: locate the gene's exon coordinates, fetch each exon from a FASTA, reverse-complement (because KRAS is on the minus strand), splice them together, apply the variant at the correct *mRNA* coordinate (not the genomic one), then translate.

It's not difficult — it's just a lot of bookkeeping that doesn't exist in SeqMat code because the data model carries it. We skip the full Biopython rewrite here; the point is the *shape* of the work, not a contest.

## Summary — when to reach for which

| Task                                                                | Better choice  |
| ------------------------------------------------------------------- | -------------- |
| Reverse-complement, translate, find ORFs on a raw string            | **Biopython**  |
| Sequence I/O (FASTA, GenBank, EMBL, alignments, BLAST, NCBI fetch)  | **Biopython**  |
| Look up the gene at a genomic coordinate                            | **SeqMat**     |
| Apply a variant at a genomic coordinate and see the protein effect  | **SeqMat**     |
| Mutate a sequence and audit "what changed and where"                | **SeqMat**     |
| Splice introns out while keeping a coordinate map for downstream    | **SeqMat**     |
| Long-sequence byte ops where you don't need the coordinate model    | **Biopython**  |

Both are pip-installable. Many real pipelines use both — Biopython for the I/O and the raw-string fast path, SeqMat where coordinate and mutation bookkeeping would otherwise be hand-written.